In [1]:
# ============================================================
# Celda 1 — Setup
# ============================================================
import pandas as pd
import numpy as np
from pathlib import Path

GOLD   = Path("output")
MERGED = Path("output/merged")
MERGED.mkdir(parents=True, exist_ok=True)
print("✅ Rutas OK")

✅ Rutas OK


In [2]:
# ============================================================
# Celda 2 — Cargar todos los gold
# ============================================================
import pandas as pd
from pathlib import Path
import os

# ROOT dinámico — funciona en local, CI y cualquier entorno
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

OUTPUT = Path("output")

evr   = pd.read_parquet(OUTPUT / "evr/03-gold/gold_evr_provincia_anio.parquet")
eco_p = pd.read_parquet(OUTPUT / "economico/03-gold/gold_economico_provincia_anio.parquet")
con   = pd.read_parquet(OUTPUT / "conectividad/03-gold/gold_conectividad_provincia_anio.parquet")

# Normalizar nombres de columna clave
con = con.rename(columns={'ccaa': 'ccaa_conectividad'})

print(f"EVR:          {evr.shape}  | años {evr['anyo'].min()}→{evr['anyo'].max()}")
print(f"Económico:    {eco_p.shape}  | años {eco_p['anyo'].min()}→{eco_p['anyo'].max()}")
print(f"Conectividad: {con.shape}  | años {con['anyo'].min()}→{con['anyo'].max()}")

EVR:          (728, 8)  | años 2008→2021
Económico:    (4056, 7)  | años 1996→2024
Conectividad: (624, 6)  | años 2013→2024


In [3]:
# ============================================================
# Celda 3 — Merge provincia × año (base = EVR 2008-2021)
# ============================================================
df = evr.copy()

# Join económico
df = pd.merge(df, eco_p, on=['provincia', 'anyo'], how='left')

# Join conectividad (solo años 2013-2021 tendrán datos)
df = pd.merge(df, con[['provincia','anyo','cob_100mbps_pct','ranking_anio']],
              on=['provincia','anyo'], how='left')

df = df.sort_values(['provincia', 'anyo']).reset_index(drop=True)

print(f"Shape merged: {df.shape}")
print(f"Columnas ({len(df.columns)}): {list(df.columns)}")
print(f"\nCobertura temporal por columna (% no nulos):")
cov = (df.notnull().sum() / len(df) * 100).round(1)
print(cov.to_string())

Shape merged: (2184, 15)
Columnas (15): ['provincia', 'anyo', 'entradas', 'salidas', 'saldo_neto', 'tasa_atraccion', 'delta_saldo_yoy', 'ranking_atraccion_anio', 'compraventas_vivienda', 'poblacion_provincia', 'viajeros', 'pernoctaciones', 'compraventas_per_capita', 'cob_100mbps_pct', 'ranking_anio']

Cobertura temporal por columna (% no nulos):
provincia                  100.0
anyo                       100.0
entradas                   100.0
salidas                    100.0
saldo_neto                 100.0
tasa_atraccion             100.0
delta_saldo_yoy             92.9
ranking_atraccion_anio     100.0
compraventas_vivienda      100.0
poblacion_provincia        100.0
viajeros                    82.7
pernoctaciones              82.7
compraventas_per_capita    100.0
cob_100mbps_pct             63.0
ranking_anio                63.0


In [4]:
#============================================================
# Celda 4 — Guardar merged
#============================================================
MERGED = Path("output/merged")
MERGED.mkdir(parents=True, exist_ok=True)   # ← esto faltaba

df.to_parquet(MERGED / "master_provincia_anio.parquet", index=False)
df.to_csv(    MERGED / "master_provincia_anio.csv",     index=False)

print(f"✅ Master guardado: {df.shape}")
print(f"\nMuestra Madrid:")
print(df[df['provincia']=='Madrid'].to_string(index=False))

✅ Master guardado: (2184, 15)

Muestra Madrid:
provincia  anyo  entradas  salidas  saldo_neto  tasa_atraccion  delta_saldo_yoy  ranking_atraccion_anio  compraventas_vivienda  poblacion_provincia  viajeros  pernoctaciones  compraventas_per_capita  cob_100mbps_pct  ranking_anio
   Madrid  2008     17936    79288      -61352         -3.4206             <NA>                      52               177882.0            6271638.0       NaN             NaN                 0.028363              NaN           NaN
   Madrid  2008     17936    79288      -61352         -3.4206             <NA>                      52               177882.0            3040658.0       NaN             NaN                 0.058501              NaN           NaN
   Madrid  2008     17936    79288      -61352         -3.4206             <NA>                      52               177882.0            3230980.0       NaN             NaN                 0.055055              NaN           NaN
   Madrid  2009     10071    7349